In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
import pathlib
import numpy as np

In [19]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [20]:
class SoCalGuessrDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = pathlib.Path(folder_path)
        self.transform = transform
        
        # Grab all the .jpg files in the folder
        self.image_paths = list(self.folder_path.glob("*.jpg"))
        
        # The exact same mapping used for your custom CNN
        self.classes = ['Los_Angeles', 'San_Diego', 'SLO', 'Bakersfield', 'Riverside', 'Anaheim']
        self.city_to_idx = {city: idx for idx, city in enumerate(self.classes)}
    
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        # Load the image and force RGB
        image = Image.open(img_path).convert('RGB')
        
        # Extract the label from the filename (e.g., "Anaheim-123.jpg" -> "Anaheim")
        city_name = img_path.name.split('-')[0]
        label = self.city_to_idx[city_name]
        
        # Apply the ResNet transforms
        if self.transform:
            image = self.transform(image)
            
        return image, label
    
    # --- 3. The 80/20 Train & Validation Split ---
folder_dir = './final_images'

# Instantiate the dataset twice to apply the different transforms
full_train_dataset = SoCalGuessrDataset(folder_path=folder_dir, transform=train_transform)
full_val_dataset = SoCalGuessrDataset(folder_path=folder_dir, transform=val_transform)

# Generate a shuffled list of indices
num_images = len(full_train_dataset)
indices = np.random.permutation(num_images)
split_point = int(0.8 * num_images)

# Split the indices
train_indices = indices[:split_point]
val_indices = indices[split_point:]

# Assign the indices to the correct transform rules
#change loader to train_subset
train_subset = Subset(full_train_dataset, train_indices)
val_subset = Subset(full_val_dataset, val_indices)

# --- 4. Create the DataLoaders ---
batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training images: {len(train_subset)}")
#changed from train_subset
print(f"Validation images: {len(val_subset)}")

Training images: 7344
Validation images: 1837


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 6)
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)




Using device: cpu


In [19]:
epochs = 15 # Transfer learning usually requires far fewer epochs!

train_loss_history = []
val_loss_history = []
train_acc_history = []
val_acc_history = []

for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        pred = model(X_batch)
        loss = loss_fn(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        predicted_classes = torch.argmax(pred, dim=1)
        correct += (predicted_classes == y_batch).sum().item()
        n += len(y_batch)
        
    epoch_train_loss = total_loss / n
    epoch_train_acc = correct / n
    train_loss_history.append(epoch_train_loss)
    train_acc_history.append(epoch_train_acc)

    # --- Evaluation Phase ---
    model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0
    
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            
            val_pred = model(X_val)
            v_loss = loss_fn(val_pred, y_val)
            
            val_loss += v_loss.item() * len(y_val)
            val_predicted_classes = torch.argmax(val_pred, dim=1)
            val_correct += (val_predicted_classes == y_val).sum().item()
            val_n += len(y_val)
            
    epoch_val_loss = val_loss / val_n
    epoch_val_acc = val_correct / val_n
    val_loss_history.append(epoch_val_loss)
    val_acc_history.append(epoch_val_acc)
    
    print(f"Epoch {epoch+1:2d}/{epochs}  "
          f"train_loss={epoch_train_loss:.4f}  "
          f"val_loss={epoch_val_loss:.4f}  "
          f"train_acc={epoch_train_acc:.3f}  "
          f"val_acc={epoch_val_acc:.3f}")

Epoch  1/15  train_loss=1.2225  val_loss=0.9540  train_acc=0.582  val_acc=0.683
Epoch  2/15  train_loss=0.8546  val_loss=0.7999  train_acc=0.724  val_acc=0.739
Epoch  3/15  train_loss=0.7586  val_loss=0.7451  train_acc=0.748  val_acc=0.740
Epoch  4/15  train_loss=0.7201  val_loss=0.7058  train_acc=0.760  val_acc=0.761
Epoch  5/15  train_loss=0.6837  val_loss=0.6862  train_acc=0.763  val_acc=0.761
Epoch  6/15  train_loss=0.6583  val_loss=0.6600  train_acc=0.777  val_acc=0.776
Epoch  7/15  train_loss=0.6398  val_loss=0.6468  train_acc=0.781  val_acc=0.788
Epoch  8/15  train_loss=0.6161  val_loss=0.6362  train_acc=0.790  val_acc=0.781
Epoch  9/15  train_loss=0.6097  val_loss=0.6425  train_acc=0.790  val_acc=0.780
Epoch 10/15  train_loss=0.6022  val_loss=0.6366  train_acc=0.794  val_acc=0.782
Epoch 11/15  train_loss=0.5961  val_loss=0.6149  train_acc=0.797  val_acc=0.795
Epoch 12/15  train_loss=0.5808  val_loss=0.6183  train_acc=0.803  val_acc=0.784
Epoch 13/15  train_loss=0.5661  val_loss

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
num_features_t = model.heads.head.in_features
model.heads.head = nn.Linear(num_features_t, 6)
model = model.to(device)

Using device: cpu


In [24]:
loss_fn_t = nn.CrossEntropyLoss()
#changed from adam to adamW
optimizer_t = optim.AdamW(model.heads.parameters(), lr=0.001, weight_decay=0.01)


In [17]:
epochs_t = 15
train_loss_history_t = []
val_loss_history_t = []
train_acc_history_t = []
val_acc_history_t = []
for epoch in range(epochs_t):
    print(f"--- Starting Epoch {epoch+1}/{epochs_t} ---")
    
    # --- Training Phase ---
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    
    for i, (X_batch, y_batch) in enumerate(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        pred = model(X_batch)
        loss = loss_fn_t(pred, y_batch)

        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

        #if (i + 1) % 20 == 0:
            #print(f"   Processed batch {i+1} of {len(train_loader)}...")

        total_loss += loss.item() * len(y_batch)
        predicted_classes = torch.argmax(pred, dim=1)
        correct += (predicted_classes == y_batch).sum().item()
        n += len(y_batch)
        
    epoch_train_loss = total_loss / n
    epoch_train_acc = correct / n
    train_loss_history_t.append(epoch_train_loss)
    train_acc_history_t.append(epoch_train_acc)

    # --- Evaluation Phase ---
    #print("   Evaluating validation set...")
    model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0
    
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            
            val_pred = model(X_val)
            v_loss = loss_fn_t(val_pred, y_val)
            
            val_loss += v_loss.item() * len(y_val)
            val_predicted_classes = torch.argmax(val_pred, dim=1)
            val_correct += (val_predicted_classes == y_val).sum().item()
            val_n += len(y_val)
            
    epoch_val_loss = val_loss / val_n
    epoch_val_acc = val_correct / val_n
    val_loss_history_t.append(epoch_val_loss)
    val_acc_history_t.append(epoch_val_acc)
    
    print(f"Epoch {epoch+1:2d} Summary: "
          f"train_loss={epoch_train_loss:.4f}  "
          f"val_loss={epoch_val_loss:.4f}  "
          f"train_acc={epoch_train_acc:.3f}  "
          f"val_acc={epoch_val_acc:.3f}\n")


--- Starting Epoch 1/15 ---


KeyboardInterrupt: 